## 0 Setup

In [1]:
# !pip install anthropic

In [2]:
!pip install -U brotli


[notice] A new release of pip is available: 26.1.1 -> 26.3
[notice] To update, run: C:\Users\Patrick\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [ ]:
from pathlib import Path
from dotenv import load_dotenv
import os, anthropic

ENV_PATH = Path(".env")  # copy .env.example to .env and add your key
load_dotenv(ENV_PATH, override=True)   # explicit path; .env overrides any stale env var

key = os.getenv("ANTHROPIC_API_KEY", "").strip().strip('"').strip("'")
print("key loaded:", bool(key), "| starts with sk-ant:", key.startswith("sk-ant"), "| length:", len(key))

client = anthropic.Anthropic(api_key=key)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import nltk
import anthropic
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import os
print(os.getenv("ANTHROPIC_API_KEY") is not None)

True


In [ ]:
nltk.download('vader_lexicon')

[nltk_data] Downloading package vader_lexicon to
[nltk_data]     C:\Users\Patrick\AppData\Roaming\nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


True

## 1 Import and Processing

In [ ]:
df = pd.read_csv("Tweets.csv")
df.tail()

,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
14635,569587686496825344,positive,0.3487,NaN,0.0000,American,NaN,KristenReenders,NaN,0,@AmericanAir thank you we got on a different f...,NaN,2015-02-22 12:01:01 -0800,NaN,NaN
14636,569587371693355008,negative,1.0000,Customer Service Issue,1.0000,American,NaN,itsropes,NaN,0,@AmericanAir leaving over 20 minutes Late Flig...,NaN,2015-02-22 11:59:46 -0800,Texas,NaN
14637,569587242672398336,neutral,1.0000,NaN,NaN,American,NaN,sanyabun,NaN,0,@AmericanAir Please bring American Airlines to...,NaN,2015-02-22 11:59:15 -0800,"Nigeria,lagos",NaN
14638,569587188687634433,negative,1.0000,Customer Service Issue,0.6659,American,NaN,SraJackson,NaN,0,"@AmericanAir you have my money, you change my ...",NaN,2015-02-22 11:59:02 -0800,New Jersey,Eastern Time (US & Canada)
14639,569587140490866689,neutral,0.6771,NaN,0.0000,American,NaN,daviddtwu,NaN,0,@AmericanAir we have 8 ppl so we need 2 know h...,NaN,2015-02-22 11:58:51 -0800,"dallas, TX",NaN


In [ ]:
df.shape

(14640, 15)

In [ ]:
df = df.sample(200, random_state=1).reset_index(drop=True)
df.shape

(200, 15)

In [ ]:
df = df[['airline', 'airline_sentiment', 'text']]
df.head()

,airline,airline_sentiment,text
0,Delta,positive,@JetBlue I'll pass along the advice. You guys ...
1,United,negative,@united I sent you a dm with my file reference...
2,Southwest,positive,@SouthwestAir Black History Commercial is real...
3,Southwest,negative,@SouthwestAir why am I still in Baltimore?! @d...
4,Southwest,positive,@SouthwestAir SEA to DEN. South Sound Volleyba...


In [ ]:
df['text'][0]

"@JetBlue I'll pass along the advice. You guys rock!!"

## 2 Claude API (Haiku)

In [ ]:
client = anthropic.Anthropic()

prompt = """
You classify the sentiments  of tweets written to US airlines.
Reply with exactly one word: positive, neutral, negative.
Sarcasm counts as its true sentiment (for example, "thanks for losing my bag" is negative).

Tweet: """

In [ ]:
def compute_claude_sentiment(text):
    response = client.messages.create(
        model='claude-haiku-5-5',
        max_tokens=20,
        thinking={'type': 'disabled'},
        messages=[{'role': 'user', 'content': prompt + text}]
    )

    # take the first text block, wherever it is
    answer = next((b.text for b in response.content if b.type == 'text'), '')
    answer = answer.strip(" .!\n").lower()
    return answer if answer in ['positive', 'neutral', 'negative'] else 'neutral'

In [ ]:
compute_claude_sentiment('Your Airline Sucks!')

APIConnectionError: Connection error.

## Applying Claude to Data

In [ ]:
df['claude_sentiment'] = df['text'].apply(compute_claude_sentiment)

df[['text', 'airline_sentiment', 'claude_sentiment']].head()

APIConnectionError: Connection error.

In [ ]:
df

,airline,airline_sentiment,text,claude_sentiment
0,Delta,positive,@JetBlue I'll pass along the advice. You guys ...,positive
1,United,negative,@united I sent you a dm with my file reference...,neutral
2,Southwest,positive,@SouthwestAir Black History Commercial is real...,positive
3,Southwest,negative,@SouthwestAir why am I still in Baltimore?! @d...,negative
4,Southwest,positive,@SouthwestAir SEA to DEN. South Sound Volleyba...,positive
...,...,...,...,...
195,US Airways,negative,@USAirways I just want to be heard and to disc...,neutral
196,Southwest,negative,@SouthwestAir I wrote to your customer support...,negative
197,Delta,negative,@JetBlue I've spent three hours talking to all...,negative
198,US Airways,positive,@USAirways thanks to the whole team for an on-...,positive


### Vader Sentiment (Comparison)

In [ ]:
sia = SentimentIntensityAnalyzer()

def compute_vader_sentiment(text):
    score = sia.polarity_scores(text)['compound']

    if score >= 0.05:
        return 'positive'
    elif score <= -0.05:
        return 'negative'
    return 'neutral'

In [ ]:
df['vader_sentiment'] = df['text'].apply(compute_vader_sentiment)

In [ ]:
df

,airline,airline_sentiment,text,claude_sentiment,vader_sentiment
0,Delta,positive,@JetBlue I'll pass along the advice. You guys ...,positive,neutral
1,United,negative,@united I sent you a dm with my file reference...,neutral,positive
2,Southwest,positive,@SouthwestAir Black History Commercial is real...,positive,positive
3,Southwest,negative,@SouthwestAir why am I still in Baltimore?! @d...,negative,positive
4,Southwest,positive,@SouthwestAir SEA to DEN. South Sound Volleyba...,positive,neutral
...,...,...,...,...,...
195,US Airways,negative,@USAirways I just want to be heard and to disc...,neutral,positive
196,Southwest,negative,@SouthwestAir I wrote to your customer support...,negative,positive
197,Delta,negative,@JetBlue I've spent three hours talking to all...,negative,negative
198,US Airways,positive,@USAirways thanks to the whole team for an on-...,positive,positive


## Comparison of Models

In [ ]:
pd.crosstab(df['airline_sentiment'], df['claude_sentiment'])

claude_sentiment,negative,neutral,positive
airline_sentiment,,,
negative,123,8,2
neutral,9,26,3
positive,2,2,25


In [ ]:
pd.crosstab(df['airline_sentiment'], df['vader_sentiment'])

vader_sentiment,negative,neutral,positive
airline_sentiment,,,
negative,67,29,37
neutral,8,15,15
positive,0,3,26
